In [ ]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
import numpy as np
import os
import json
from tqdm import tqdm

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
DATA_PATH = "../data/cleaned_data/merged_training_v3.csv"  # your dataset
SAVE_DIR = "models/dl_resume_match"
EPOCHS = 4
BATCH_SIZE = 16
LR = 2e-5
MAX_LEN = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
class ResumeJDDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        resume_text = str(self.df.iloc[idx]["resume_text"])
        jd_text = str(self.df.iloc[idx]["jd_text"])
        label = self.df.iloc[idx]["match_score"] / 100.0 

        enc_resume = self.tokenizer(
            resume_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        enc_jd = self.tokenizer(
            jd_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "resume_input_ids": enc_resume["input_ids"].squeeze(),
            "resume_attention_mask": enc_resume["attention_mask"].squeeze(),
            "jd_input_ids": enc_jd["input_ids"].squeeze(),
            "jd_attention_mask": enc_jd["attention_mask"].squeeze(),
            "label": torch.tensor(label, dtype=torch.float)
        }

In [ ]:
class ResumeMatchModel(nn.Module):
    def __init__(self, model_name):
        super(ResumeMatchModel, self).__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, resume_input_ids, resume_attention_mask, jd_input_ids, jd_attention_mask):
        r_out = self.encoder(input_ids=resume_input_ids, attention_mask=resume_attention_mask).pooler_output
        j_out = self.encoder(input_ids=jd_input_ids, attention_mask=jd_attention_mask).pooler_output
        combined = torch.cat((r_out, j_out), dim=1)
        return self.regressor(combined).squeeze()

In [ ]:
def train_fn(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training"):
        optimizer.zero_grad()
        outputs = model(
            batch["resume_input_ids"].to(DEVICE),
            batch["resume_attention_mask"].to(DEVICE),
            batch["jd_input_ids"].to(DEVICE),
            batch["jd_attention_mask"].to(DEVICE)
        )
        loss = criterion(outputs, batch["label"].to(DEVICE))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_fn(model, loader, criterion):
    model.eval()
    total_loss = 0
    preds, truths = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating"):
            outputs = model(
                batch["resume_input_ids"].to(DEVICE),
                batch["resume_attention_mask"].to(DEVICE),
                batch["jd_input_ids"].to(DEVICE),
                batch["jd_attention_mask"].to(DEVICE)
            )
            loss = criterion(outputs, batch["label"].to(DEVICE))
            total_loss += loss.item()
            preds.extend(outputs.cpu().numpy())
            truths.extend(batch["label"].cpu().numpy())
    return total_loss / len(loader), preds, truths

In [ ]:
if __name__ == "__main__":
    df = pd.read_csv(DATA_PATH)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

    train_ds = ResumeJDDataset(train_df, tokenizer, MAX_LEN)
    val_ds = ResumeJDDataset(val_df, tokenizer, MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    model = ResumeMatchModel(MODEL_NAME).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    for epoch in range(EPOCHS):
        train_loss = train_fn(model, train_loader, optimizer, criterion)
        val_loss, preds, truths = eval_fn(model, val_loader, criterion)
        print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, "dl_resume_match.pt"))
            print("Model saved!")

Validating: 100%|█████████████████████████████████████████████████████████████████████████████████| 64/64 [00:32<00:00,  1.99it/s]


Epoch 1/4 - Train Loss: 0.0049 - Val Loss: 0.0031
Model saved!


Validating: 100%|█████████████████████████████████████████████████████████████████████████████████| 64/64 [00:41<00:00,  1.55it/s]


Epoch 2/4 - Train Loss: 0.0029 - Val Loss: 0.0029
Model saved!


Validating: 100%|█████████████████████████████████████████████████████████████████████████████████| 64/64 [00:41<00:00,  1.55it/s]


Epoch 3/4 - Train Loss: 0.0024 - Val Loss: 0.0034


Validating: 100%|█████████████████████████████████████████████████████████████████████████████████| 64/64 [00:41<00:00,  1.55it/s]

Epoch 4/4 - Train Loss: 0.0021 - Val Loss: 0.0027
Model saved!


In [ ]:
metrics = {"best_val_loss": best_val_loss,"epochs": EPOCHS}
import json
with open(OUTPUT_DIR / "training_metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)